# Dynamic Core-Satellite Modelling with Hybrid PCA-Based Statistical Arbitrage Strategies
*You can use this provided template for your personal experimentation. **Do not** use this for your personal finances or investments. This template is provided for research purposes only, not as a proposed investment strategy. I am not responsible for anything that happens if you use this code for your personal finances.*

## Imports
*Please install and import the following packages in order to use the provided code. Note that this code was initially designed to be executed in Google Colab.*

In [ ]:
!pip install "git+https://github.com/dppalomar/pob.git#subdirectory=python"
!pip install pykalman
!pip install yfinance
!pip install cvxpy

In [ ]:
from pob_python import SP500_stocks_2015to2020, SP500_index_2015to2020
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from pykalman import KalmanFilter
import matplotlib.pyplot as plt
import yfinance as yf
from statsmodels.tsa.stattools import adfuller
import cvxpy as cp

## Data Setup

In [ ]:
# This is your data which you are using to test the model on. You can change this to whichever data you would like.

stock_prices = SP500_stocks_2015to2020[
                  ["AAPL", "AMZN", "AMD", "GM", "GOOGL", "MGM", "MSFT", "QCOM", "TSCO", "UPS"]
             ].loc["2017":"2019"]

# Convert to log-prices and drop missing values to ensure that the data is clean
log_returns = np.log(stock_prices / stock_prices.shift(1)).dropna()

In [ ]:
# If you would like to compare your experimentation results to the true market, please run the following.

data = yf.download(["AAPL", "AMZN", "AMD", "GM", "GOOGL", "MGM", "MSFT", "QCOM", "TSCO", "UPS"], start='2017-01-04', end='2019-10-31', auto_adjust=False)

# Use Adjusted Close prices
closed_data = data['Adj Close']

# Compute log returns
log_returns_sp500 = np.log(closed_data / closed_data.shift(1)).dropna()
benchmark_returns = log_returns_sp500.mean(axis=1)
benchmark_cumulative = np.exp(np.cumsum(benchmark_returns)) - 1

## Parameters
*You may adjust the provided parameters in any way you would like.*

In [ ]:
# --- Parameters ---
window = 40              # rolling window length for PCA and residuals
rebalance_freq = 5       # rebalance frequency in days
pval_threshold = 0.005   # ADF stationarity significance level
alpha_split = 0.85       # fraction of capital in baseline vs stat-arb overlay
entry_threshold = 2.0    # s-score entry threshold
exit_threshold = 0.5     # s-score exit threshold
stop_loss = -0.02        # stop-loss at -2% cumulative PnL
max_holding = 10         # maximum holding horizon in days
residual_threshold = 1.0  # minimum absolute residual size required to consider trading
stop_loss_threshold = -0.02   # stop-loss at -2% cumulative PnL
max_holding_days = 10         # maximum holding horizon

## Developed Algorithm and Strategy

In [ ]:
"""
Dynamic Core-Satellite Portfolio Algorithm with Hybrid PCA-Based Statistical Arbitrage Strategies
Core-Satellite Portfolio Algorithm with PCA Residuals and Statistical Arbitrage Signals

Overview:
---------
This algorithm constructs a dynamic core-satellite portfolio that blends a
baseline equal-weighted "core" allocation with an adaptive "satellite"
allocation derived from statistical arbitrage signals. The satellite signals
are generated using PCA residuals, Kalman smoothing, robust z-scores, and
stationarity testing. Risk controls (stop-loss, capped satellite fraction,
maximum holding days) are embedded directly into the weight update loop.

Key Features:
-------------
- Core portfolio: equal-weighted baseline allocation. --> This will be the part you replace with your experimental portfolio
- Satellite portfolio: PCA residual-based mean-reversion signals.
- Dynamic blending: adaptive core/satellite split based on signal strength
  and relative risk.
- Risk management: stop-loss, holding period limits, and exposure caps.
- Output: cumulative strategy returns compared against market baseline.

Key Contributions:
------------------
1. PCA residual extraction isolates idiosyncratic signals.
2. Kalman filtering smooths residuals for robust signal generation.
3. Dynamic core-satellite blending improves risk-adjusted performance.

Implementation Details:
-----------------------
- Uses rolling windows up to t-1 to ensure no lookahead bias
- PCA is used for residual extraction
- THe Kalman smoothing is fit on only past data
- S-score used to standardize signals with heavy-tail estimators suited for financial data
- Both stationarity testing, and contrarian entry/exit rules are applied
- Risk controls:
    * Stop-loss: exit if cumulative PnL < threshold
    * Max holding horizon: exit after N days if no reversion
- Tracking structures: position_pnl and position_days
- Blended allocation with core-sattelite modelling with the selected baseline portfolio

Outputs:
--------
- strategy_series: time series of portfolio returns
- strategy_cum: cumulative portfolio performance
"""


# --- Tracking structures ---
position_pnl = {}        # cumulative PnL per asset
position_days = {}       # holding days per asset

# --- Strategy loop ---
strategy_returns = []
current_weights = np.ones(log_returns.shape[1]) / log_returns.shape[1]  # safe init

for t in range(window, len(log_returns)):
    if (t - window) % rebalance_freq == 0:
        # === Rolling window data (up to t-1) ===
        window_data = log_returns.iloc[t-window:t-1, :]

        """
        Step 1: Baseline Portfolio Construction
        ---------------------------------------------
        Select or build a vanilla baseline portfolio.
        It should specifically be designed to provide the strategy with diversification.
        """
        # TODO: Implement your selected portfolio
        base_weights = # TODO: Calculate the baseweights from your portfolio
        base_weights /= np.sum(np.abs(base_weights)) # This is to normalize by absolute sum

        """
        Step 2: Extraction of PCA residuals from the rollowing window data
        -------------------------------------------------------------------
        This standardizes log returns, run PCA, reconstruct ~80% (which you can change if you would like)
        of variance, and compute residuals (idiosyncratic deviations). Then it produces the residuals,
        which represents the idiosyncratic components of returns after PCA reconstruction.

        Notes
        -----
        - PCA is applied to standardized returns.
        - Residuals represent deviations orthogonal to retained components.
        - These residuals form the basis of satellite trading signals.

        Parameters in this section
        --------------------------
        window_data : pd.DataFrame
          Standardized log returns for the rolling window.

         Final values calculated in this section
        ---------------------------------------
        residuals : np.ndarray
          Idiosyncratic components of returns after PCA reconstruction.
        n_components_80 : int
          Number of principal components retained.
        """
        scaler = StandardScaler()
        scaled_window = scaler.fit_transform(window_data)

        # Fit PCA to full window
        pca_full = PCA()
        pca_full.fit(scaled_window)

        # Determine number of components explaining ≥80% variance
        cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
        n_components_80 = np.argmax(cumulative_variance >= 0.80) + 1

        # Reconstruct data using retained components
        pca = PCA(n_components=n_components_80)
        scores = pca.fit_transform(scaled_window)

        # Residuals = idiosyncratic components
        reconstructed = pca.inverse_transform(scores)
        residuals = scaled_window - reconstructed

        """
        Step 3: Kalman Filter Smoothing
        --------------------------------
        This step is to apply Kalman filtering to residuals for noise reduction.
        This is in order to produce smooth residuals which reduce noise and estimate latent states.
        This is done for each asset.
        However, in case of filter failure, the raw residuals are used.

        Notes
        -----
        - Kalman filter estimates latent states dynamically.
        - Provides smoother signals compared to raw residuals.
        - Fallback to last residual if Kalman fails.

        Parameters in this section
        --------------------------
        residuals : np.ndarray
          Residual time series for each asset.

         Final values calculated in this section
        ---------------------------------------
        smoothed_residuals : np.ndarray
          Final smoothed residual values for each asset.
        """
        smoothed_residuals = []
        for i in range(residuals.shape[1]):
            series = residuals[:, i].reshape(-1, 1)
            try:
                # Estimate latent state with Kalman filter
                kf = KalmanFilter(initial_state_mean=0, n_dim_obs=1)
                state_means, _ = kf.em(series).filter(series)
                smoothed_residuals.append(state_means[-1][0])
            except Exception:
                # This line is that which provides the fallback: use last residual if Kalman fails
                smoothed_residuals.append(residuals[-1, i])
        smoothed_residuals = np.array(smoothed_residuals)

        """
        Step 4: Robust z-score calculation for residuals
        ------------------------------------------------
        This standardizes the smoothed residuals using median and MAD for a robust approach:
        S_i = (R_i - μ) / σ

        Notes
        -----
        - Resistant to outliers and heavy-tailed distributions.
        - Scores indicate deviation from median behavior: Large deviations indicate potential mean-reversion trades.

        Parameters in this section
        --------------------------
        residuals : np.ndarray
            Residual time series for each asset.
        smoothed_residuals : np.ndarray
            Smoothed residual values for each asset.

         Final values calculated in this section
        ---------------------------------------
        s_scores : np.ndarray
          Robust z-scores for each asset.
        """
        s_scores = []
        for i in range(residuals.shape[1]):
            mu = np.median(residuals[:, i])
            sigma = max(np.median(np.abs(residuals[:, i] - mu)) * 1.4826, 1e-6)  # This is to provide a stability floor, in case it becomes too small
            s_scores.append((smoothed_residuals[i] - mu) / sigma)
        s_scores = np.array(s_scores)

        """
        Step 5: Generate statistical arbitrage trading signals from PCA residuals with stationarity testing
        ----------------------------------------------------------------------------------------------------
        Accepts the residuals and s-scores for each asset to determine whether each should be traded according
        to the different thresholds and current positions.

        - Entry: If s > +2, short; if s < -2, long. --> These values can be changed if you would like
        - Exit: Close when |s| < 0.5 (reversion).
        - Only trade residuals that pass ADF stationarity test.

        Notes
        -----
        - Signals are only generated for residuals that pass the ADF stationarity test.
        - Long positions are entered when z-scores fall below -entry_threshold.
        - Short positions are entered when z-scores exceed +entry_threshold.
        - Positions are exited when z-scores revert within exit_threshold.
        - This ensures trades are based on mean-reverting processes rather than spurious trends.

        Parameters in this section
        --------------------------
        residuals : np.ndarray
            Residual time series for each asset.
        s_scores : np.ndarray
            Robust z-scores for each asset, indicating deviation from median behavior.
        pval_threshold : float
            Significance threshold for ADF stationarity test.
        entry_threshold : float
            Z-score threshold for entering long/short positions.
        exit_threshold : float
            Z-score threshold for exiting positions.
        position_days : dict
            Dictionary tracking the number of days each position has been held.

         Final values calculated in this section
        ----------------------------------------
        statarb_weights : np.ndarray
          Vector of trading signals (-1 = short, +1 = long, 0 = flat).
        position_days : dict
          Updated dictionary of holding durations.
        """
        statarb_weights = np.zeros(residuals.shape[1])
        for i in range(residuals.shape[1]):
            try:
                # Stationarity test (ADF)
                series = residuals[:, i]
                pval = adfuller(series)[1]
            except Exception:
                pval = 1.0  # treat as non-stationary

            # Trading rules based on z-scores and stationarity
            if pval < pval_threshold:
                if s_scores[i] > entry_threshold:
                    statarb_weights[i] = -1 # short signal
                    position_days[i] = position_days.get(i, 0) + 1
                elif s_scores[i] < -entry_threshold:
                    statarb_weights[i] = +1 # long signal
                    position_days[i] = position_days.get(i, 0) + 1
                elif abs(s_scores[i]) < exit_threshold:
                    statarb_weights[i] = 0 # exit signal
                    position_days[i] = 0

        """
        Step 6: Risk Controls and Risk Management Constraints to StatArb positions
        --------------------------------------------------------------------------
        This is to enforce stop-loss and maximum holding horizon.
        From this, the StatArb weights for the satelite will be determined.

        Notes
        -----
        - Stop-loss: positions are closed if cumulative PnL falls below threshold.
        - Max holding period: positions are closed if held beyond allowed duration.
        - Normalization: active positions are rescaled to maintain balanced exposure.
        - These controls prevent runaway losses and enforce disciplined trading.

        Parameters in this section
        --------------------------
        statarb_weights : np.ndarray
            Current trading signals (-1 = short, +1 = long, 0 = flat).
        log_returns : pd.DataFrame
            Log returns of assets.
        t : int
            Current time index.
        position_pnl : dict
            Dictionary tracking cumulative PnL for each position.
        position_days : dict
            Dictionary tracking the number of days each position has been held.
        stop_loss_threshold : float
            Maximum allowable loss before a position is closed.
        max_holding_days : int
            Maximum number of days a position can be held.

        Final values calculated in this section
        ---------------------------------------
        statarb_weights : np.ndarray
            Updated trading signals after risk controls.
        position_pnl : dict
            Updated cumulative PnL tracking.
        position_days : dict
            Updated holding durations.
        """
        for i in range(len(statarb_weights)):
            if statarb_weights[i] != 0:
                asset_return = log_returns.iloc[t, i]
                position_pnl[i] = position_pnl.get(i, 0.0) + statarb_weights[i] * asset_return

                # Stop-loss enforcement
                if position_pnl[i] < stop_loss_threshold:
                    statarb_weights[i] = 0
                    position_pnl[i] = 0.0
                    position_days[i] = 0

                # Max holding period enforcement
                if position_days.get(i, 0) >= max_holding_days:
                    statarb_weights[i] = 0
                    position_pnl[i] = 0.0
                    position_days[i] = 0

        # Normalize satellite weights if active positions exist
        if np.sum(np.abs(statarb_weights)) > 0:
            statarb_weights = statarb_weights / np.sum(np.abs(statarb_weights))

        """
        Step 7: Blended Core-Satellite Allocation
        ------------------------------------------
        Compute dynamic blended allocation between core and satellite portfolios.
        Final weights are computed as:
        w_t = f_core * w_core + f_satellite * w_satellite

        This blending mechanism ensures adaptive exposure: more satellite allocation when
        signals are strong and stable, less when signals are weak or risky.

        Notes
        -----
        - Satellite fraction is scaled by average signal strength (mean |z-score|).
        - Satellite exposure is capped at max_satellite_fraction to prevent over-concentration.
        - Relative risk adjustment: if satellite risk (std. dev. of weights) exceeds core risk,
          the satellite fraction is reduced proportionally.

        Parameters in this section
        --------------------------
        base_weights : np.ndarray
          Equal-weighted baseline "core" portfolio allocation.
        statarb_weights : np.ndarray
            Statistical arbitrage "satellite" allocation derived from PCA residual signals.
        s_scores : np.ndarray
            Robust z-scores indicating average signal strength across assets.
        max_satellite_fraction : float, optional
            Maximum allowable fraction allocated to satellite portfolio (here 0.3). --> You can change this

        Final values calculated in this section
        ---------------------------------------
        current_weights : np.ndarray
            Final blended portfolio weights combining core and satellite allocations.
        core_fraction : float
            Fraction of portfolio allocated to the core.
        satellite_fraction : float
            Fraction of portfolio allocated to the satellite.
        """

        # Dynamic alpha_split calculation
        signal_strength = np.mean(np.abs(s_scores)) # average deviation
        risk_satellite = np.std(statarb_weights)    # proxy for satellite risk
        risk_core = np.std(base_weights)

        max_satellite_fraction = 0.3 # cap at 30% --> You can change this if you would like to
        satellite_fraction = min(max_satellite_fraction, signal_strength / 5.0) # You can change 5.0 for different signal_strengths

        # djust satellite fraction if risk is higher than core for risk adjustment
        if risk_satellite > risk_core:
            satellite_fraction *= risk_core / risk_satellite

        core_fraction = 1 - satellite_fraction

        # Final blended weights
        current_weights = core_fraction * base_weights + satellite_fraction * statarb_weights

    """
    Step 8: Portfolio Returns & Performance Tracking
    ------------------------------------------------
    Apply weights to daily returns, update strategy PnL,
    and compute cumulative log returns.
    """
    actual_returns = log_returns.iloc[t, :].values
    strategy_returns.append(np.dot(current_weights, actual_returns))

# --- Convert to Series ---
dates = log_returns.index[window:]
strategy_series = pd.Series(strategy_returns, index=dates)

# --- Compute cumulative returns (log returns assumed) ---
strategy_cum = np.exp(np.cumsum(strategy_series)) - 1

## Compare the strategy with the true market

In [ ]:
# This part will calculate the annualized sharpe ratio and volatility, and plot the graph of the strategy versus the true market

# Final performance series
dates = log_returns.index[window:]
strategy_series = pd.Series(strategy_returns, index=dates)

# Compute cumulative returns (log returns assumed)
strategy_cum = np.exp(np.cumsum(strategy_series)) - 1

# Calculate volatility (annualized)
volatility = np.std(strategy_series) * np.sqrt(252)

# Calculate Sharpe ratio (annualized, assuming risk-free rate = 0)
sharpe_ratio = (np.mean(strategy_series) * 252) / volatility

print("Annualized Volatility:", round(volatility, 4))
print("Annualized Sharpe Ratio:", round(sharpe_ratio, 4))

# Plot cumulative performance
plt.figure(figsize=(10,6))
strategy_cum.plot(label="Proposed Strategy")
benchmark_cumulative.plot(label="True Market")
plt.title("Cumulative Returns: Proposed Strategy vs True Market")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# This part will calculate and compare the drawdowns of the strategy and the true market and plot the graph

# Strategy drawdowns
running_max = np.maximum.accumulate(strategy_cum)
drawdown = (strategy_cum - running_max) / running_max

# True market drawdowns
running_max3 = np.maximum.accumulate(benchmark_cumulative)
drawdown_3 = (benchmark_cumulative - running_max3) / running_max3
drawdown_3 = drawdown_3["2019"]

# Plot drawdown graph
plt.figure(figsize=(10,6))
drawdown.plot(color="red", label="Proposed Strategy")
drawdown_3.plot(color = "green", label = "True Market")
plt.title("Drawdown Comparison: Proposed Strategy vs True Market")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.legend()
plt.grid(True)
plt.show()

## Compare the strategy with the baseline portfolio alone

In [ ]:
# TODO: Add your portfolio here so you can test the vanilla version of your baseline portfolio.
# This will run it over the data and give the sharpe ratio and volatility for comparison

portfolio_returns_baseline = []

for t in range(window, len(log_returns)):
    if (t - window) % rebalance_freq == 0:
        # Rolling window data (up to t-1, inclusive)
        window_data = log_returns.iloc[t-window:t-1, :]

        # TODO: Implement your selected portfolio
        base_weights = # TODO: Calculate the baseweights from your portfolio

        current_weights = base_weights.copy()

    # Apply weights to today’s returns
    actual_returns = log_returns.iloc[t, :].values
    portfolio_returns_baseline.append(np.dot(current_weights, actual_returns))

# Convert to Series
dates = log_returns.index[window:]
baseline_series = pd.Series(portfolio_returns_baseline, index=dates)

# Compute cumulative returns (log returns assumed)
baseline_cum = np.exp(np.cumsum(baseline_series)) - 1

# Final performance series
dates = log_returns.index[window:]

# Calculate volatility (annualized)
volatility = np.std(baseline_series) * np.sqrt(252)

# Calculate Sharpe ratio (annualized, assuming risk-free rate = 0)
sharpe_ratio = (np.mean(baseline_series) * 252) / volatility

print("Baseline Annualized Volatility:", round(volatility, 4))
print("Baseline Annualized Sharpe Ratio:", round(sharpe_ratio, 4))


In [ ]:
# This will plot the graph of cumulative returns the strategy versus the vanilla portfolio

plt.figure(figsize=(10,6))
strategy_cum.plot(label="Proposed Strategy")
baseline_cum.plot(label="Baseline Algorithm")
plt.title("Cumulative Returns: Proposed Strategy vs True Market")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# This part will calculate and compare the drawdowns of the strategy and the vanilla portfolio

running_max = np.maximum.accumulate(strategy_cum)
drawdown = (strategy_cum - running_max) / running_max

running_max2 = np.maximum.accumulate(baseline_cum)
drawdown_2 = (baseline_cum - running_max2) / running_max2

In [ ]:
# This will plot drawdown graph of the proposed strategy versus the baseline portfolio alone for comparison
plt.figure(figsize=(10,6))
drawdown.plot(color="red", label="Proposed Strategy")
drawdown_2.plot(color="blue", label="Baseline")
plt.title("Drawdown Comparison: Proposed Strategy vs Baseline")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.legend()
plt.grid(True)
plt.show()